In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
print("Đã kết nối Google Drive thành công!")



Mounted at /content/drive
Đã kết nối Google Drive thành công!


In [ ]:
# Kiểm tra xem đã nhận GPU T4 chưa
!nvidia-smi

# Clone hoặc cập nhật mã nguồn dự án
import os
if not os.path.exists("/content/vi-sub-corretion"):
    !git clone https://github.com/lhhoang5425/vi-sub-corretion.git /content/vi-sub-corretion
else:
    %cd /content/vi-sub-corretion
    !git pull

%cd /content/vi-sub-corretion
!pip install -r requirements.txt -q
!pip install -q openpyxl
print("Môi trường và thư viện đã sẵn sàng!")


Mon Nov  2 04:35:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import json
from pathlib import Path
from datasets import load_dataset
from src.data_processing.noise_generator import ASRNoiseGenerator

# 1. Chọn số lượng mẫu muốn train
NUM_SAMPLES = 100000

print(f"⏳ Đang tải {NUM_SAMPLES} câu từ Hugging Face (ShynBui/Vietnamese_spelling_error)...")
raw_dataset = load_dataset("ShynBui/Vietnamese_spelling_error", split=f"train[:{NUM_SAMPLES}]")

# 2. Khởi tạo bộ tạo nhiễu ASR (chuyên trị lỗi ngọng ASR, l/n, d/gi/r, mất dấu)
generator = ASRNoiseGenerator()

Path("data/synthetic").mkdir(parents=True, exist_ok=True)
train_file = "data/synthetic/train.jsonl"
val_file = "data/synthetic/val.jsonl"

train_count = 0
val_count = 0

print(" Đang xử lý dữ liệu và tạo cặp Noisy -> Clean...")
with open(train_file, "w", encoding="utf-8") as f_train, \
     open(val_file, "w", encoding="utf-8") as f_val:

    for i, item in enumerate(raw_dataset):
        clean_text = item["text"].strip()
        if len(clean_text) < 5:
            continue

        # Tạo nhiễu ASR chân thực cho phụ đề video
        noisy_text, errs = generator.apply_noise(clean_text)

        row = json.dumps({
            "id": i + 1,
            "noisy": noisy_text,
            "clean": clean_text,
            "errors": errs
        }, ensure_ascii=False) + "\n"

        # Chia 90% train (9,000 câu) và 10% validation (1,000 câu)
        if i % 10 == 0:
            f_val.write(row)
            val_count += 1
        else:
            f_train.write(row)
            train_count += 1

print(f"🎉 HOÀN TẤT TRONG 5 GIÂY!")
print(f"📊 Tập Train     : {train_count:,} câu")
print(f"📊 Tập Validation: {val_count:,} câu")


⏳ Đang tải 100000 câu từ Hugging Face (ShynBui/Vietnamese_spelling_error)...


README.md:   0%|          | 0.00/2.82k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 48.8MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/422209 [00:00<?, ? examples/s]

 Đang xử lý dữ liệu và tạo cặp Noisy -> Clean...
🎉 HOÀN TẤT TRONG 5 GIÂY!
📊 Tập Train     : 90,000 câu
📊 Tập Validation: 10,000 câu


In [ ]:
!python -m src.models.train_seq2seq \
    --model_name "bmd1905/vietnamese-correction-v2" \
    --train_file "data/synthetic/train.jsonl" \
    --val_file "data/synthetic/val.jsonl" \
    --output_dir "checkpoints/bmd1905-sub-corrector" \
    --epochs 3 \
    --batch_size 16 \
    --lr 3e-5
        # --model_name "vinai/bartpho-syllable" \



=== BẮT ĐẦU HUẤN LUYỆN: bmd1905/vietnamese-correction-v2 ===
Thiết bị: cuda
config.json: 100% 961/961 [00:00<00:00, 4.82MB/s]
tokenizer_config.json: 100% 1.17k/1.17k [00:00<00:00, 3.68MB/s]

sentencepiece.bpe.model: downloading bytes:  28% 1.41M/5.07M [00:00<00:02, 1.68MB/s]
sentencepiece.bpe.model: downloading bytes: 100% 3.52M/3.52M [00:00<00:00, 3.85MB/s,  342kB/s  ]
sentencepiece.bpe.model: reconstructing file: 100% 5.07M/5.07M [00:00<00:00, 5.55MB/s,  493kB/s  ]
dict.txt: 100% 360k/360k [00:00<00:00, 29.5MB/s]
special_tokens_map.json: 100% 279/279 [00:00<00:00, 1.27MB/s]

model.safetensors: downloading bytes:   0% 0.00/1.58G [00:00<?, ?B/s]
model.safetensors: downloading bytes:  17% 262M/1.58G [00:01<00:03, 344MB/s, 21.0MB/s  ]
model.safetensors: reconstructing file:   6% 101M/1.58G [00:01<00:19, 76.5MB/s, 12.9kB/s  ]
model.safetensors: downloading bytes:  34% 541M/1.58G [00:02<00:02, 361MB/s, 46.0MB/s  ]
model.safetensors: reconstructing file:  13% 201M/1.58G [00:02<00:14, 98.2MB

In [ ]:
!python -m src.evaluation.run_benchmark \
    --test_file data/gold_test/sample_test.jsonl \
    --model_name_or_path checkpoints/bmd1905-sub-corrector \
    --output_excel reports/benchmark_results.xlsx \
    --output_report reports/benchmark_summary.md \
    --batch_size 16

BẮT ĐẦU CHẠY ĐÁNH GIÁ BENCHMARK ĐỐI ĐẦU CHO ĐỀ TÀI NLP-28
- Tập dữ liệu test : data/gold_test/sample_test.jsonl
- Mô hình Seq2Seq  : checkpoints/bmd1905-sub-corrector
- Số lượng mẫu test: 25 câu

[1/3] Đang đánh giá Baseline 1: Identity (giữ nguyên)...
      -> BLEU: 0.0 | WER: 92.31% | Edit-F1: 0.0%
[2/3] Đang đánh giá Baseline 2: Rule & Dictionary...
      -> BLEU: 7.5 | WER: 90.15% | Edit-F1: 0.0%
[3/3] Đang đánh giá Seq2Seq (checkpoints/bmd1905-sub-corrector)...
[Seq2SeqCorrector] Đang tải mô hình: checkpoints/bmd1905-sub-corrector trên cuda...
Loading weights: 100% 516/516 [00:00<00:00, 4757.97it/s]
[Seq2SeqCorrector] Khởi tạo mô hình thành công!
      -> BLEU: 27.09 | WER: 14.77% | Edit-F1: 43.75%

### Bảng kết quả đánh giá Benchmark định lượng (Tập Test: 25 mẫu)

| Phương pháp đối chứng | BLEU ↑ | WER (%) ↓ | Edit-P (%) ↑ | Edit-R (%) ↑ | Edit-F1 (%) ↑ |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Baseline 1 (Identity)** | **0.00** | **92.31%** | 0.00% | 0.00% | **0.00%

In [ ]:
import shutil
from pathlib import Path

%cd /content/vi-sub-corretion

drive_dir = Path("/content/drive/MyDrive/vi-sub-correction/checkpoints")
drive_dir.mkdir(parents=True, exist_ok=True)

# Tự động tìm số phiên bản tiếp theo: v1, v2, v3...
v = 1
while (drive_dir / f"bmd1905_checkpoint_v{v}.zip").exists():
    v += 1
zip_name = f"bmd1905_checkpoint_v{v}.zip"

# Nén và copy thẳng sang Google Drive
!zip -r {zip_name} checkpoints/bmd1905-sub-corrector
shutil.move(zip_name, drive_dir / zip_name)

print(f"🎉 ĐÃ LƯU THÀNH CÔNG: {zip_name} vào Google Drive!")

/content/vi-sub-corretion
⏳ Đang nén thư mục mô hình bmd1905-sub-corrector...
  adding: checkpoints/bmd1905-sub-corrector/ (stored 0%)
  adding: checkpoints/bmd1905-sub-corrector/tokenizer_config.json (deflated 75%)
  adding: checkpoints/bmd1905-sub-corrector/model.safetensors (deflated 7%)
  adding: checkpoints/bmd1905-sub-corrector/generation_config.json (deflated 41%)
  adding: checkpoints/bmd1905-sub-corrector/sentencepiece.bpe.model (deflated 49%)
  adding: checkpoints/bmd1905-sub-corrector/dict.txt (deflated 59%)
  adding: checkpoints/bmd1905-sub-corrector/training_args.bin (deflated 54%)
  adding: checkpoints/bmd1905-sub-corrector/config.json (deflated 59%)
  adding: checkpoints/bmd1905-sub-corrector/checkpoint-11250/ (stored 0%)
  adding: checkpoints/bmd1905-sub-corrector/checkpoint-11250/rng_state.pth (deflated 26%)
  adding: checkpoints/bmd1905-sub-corrector/checkpoint-11250/trainer_state.json (deflated 81%)
  adding: checkpoints/bmd1905-sub-corrector/checkpoint-11250/tokeniz

In [ ]:
!ls -lh /content/drive/MyDrive/vi-sub-correction/checkpoints/

total 19G
-rw------- 1 root root 9.5G Oct 25 16:37 bartpho_checkpoint.zip
-rw------- 1 root root 9.5G Nov  2 08:11 bmd1905_checkpoint.zip


In [ ]:
from pathlib import Path
from src.models.inference import Seq2SeqCorrector
from src.srt_utils.srt_processor import SRTSubtitleProcessor

# 1. Nạp model
corrector = Seq2SeqCorrector(model_name_or_path="checkpoints/bmd1905-sub-corrector")
processor = SRTSubtitleProcessor(corrector_model=corrector)

# 2. Tách riêng thư mục Input và Output trên Google Drive
drive_input_dir = Path("/content/drive/MyDrive/vi-sub-correction/input")
drive_output_dir = Path("/content/drive/MyDrive/vi-sub-correction/output")
drive_output_dir.mkdir(parents=True, exist_ok=True)

# Chỉ quét file gốc trong input (bỏ qua nếu lỡ có file _cleaned)
real_files = [f for f in list(drive_input_dir.glob("*.srt")) + list(drive_input_dir.glob("*.txt")) if "_cleaned" not in f.name]

if real_files:
    for file_path in real_files:
        print(f"\n⏳ ĐANG XỬ LÝ: {file_path.name}")

        # Tự động tăng số _cleaned_1, _cleaned_2... và lưu vào folder OUTPUT
        v = 1
        while (drive_output_dir / f"{file_path.stem}_cleaned_{v}{file_path.suffix}").exists():
            v += 1
        output_file = drive_output_dir / f"{file_path.stem}_cleaned_{v}{file_path.suffix}"

        # Xử lý sửa lỗi phụ đề
        if file_path.suffix.lower() == ".srt":
            processor.process_srt_file(input_srt_path=file_path, output_srt_path=output_file)
        else:
            lines = file_path.read_text(encoding="utf-8").splitlines()
            cleaned_lines = corrector.predict_batch(lines)
            output_file.write_text("\n".join(cleaned_lines) + "\n", encoding="utf-8")

        print(f"🎉 ĐÃ SỬA XONG! File lưu tại: output/{output_file.name}")
        preview = output_file.read_text(encoding="utf-8").splitlines()[:8]
        print("\n".join(preview))
        print("=" * 60)
else:
    print("⚠️ Chưa thấy file nào trong thư mục input của Google Drive!")

[Seq2SeqCorrector] Đang tải mô hình: checkpoints/bmd1905-sub-corrector trên cuda...


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

[Seq2SeqCorrector] Khởi tạo mô hình thành công!

ĐANG XỬ LÝ FILE PHỤ ĐỀ VIDEO THẬT: kichban_2026-10-25.srt
[SRTProcessor] Đã lưu phụ đề đã sửa tại: /content/drive/MyDrive/vi-sub-correction/input/kichban_2026-10-25_cleaned.srt
ĐÃ SỬA XONG! File sạch đã được lưu tại Drive: kichban_2026-10-25_cleaned.srt

--- XEM THỬ KẾT QUẢ ĐÃ SỬA TRONG FILE ---
1
00:00:04,280 --> 00:00:08,539
Xin chào tất cả quý vị và các bạn cùng đến với chương trình tổ chức 247.

2
00:00:08,880 --> 00:00:13,199
Ngày hôm nay thì câu chuyện học làm dầu có giá lên tới 1,4 tỷ

3
00:00:13,500 --> 00:00:17,739
Nhưng giảm xuống còn có 300 triệu đang trở thành tâm điểm của mạng xã hội.


ĐANG XỬ LÝ FILE PHỤ ĐỀ VIDEO THẬT: kichban_2026-10-25_cleaned_cleaned.srt
[SRTProcessor] Đã lưu phụ đề đã sửa tại: /content/drive/MyDrive/vi-sub-correction/input/kichban_2026-10-25_cleaned_cleaned_cleaned.srt
ĐÃ SỬA XONG! File sạch đã được lưu tại Drive: kichban_2026-10-25_cleaned_cleaned_cleaned.srt

--- XEM THỬ KẾT QUẢ ĐÃ SỬA TRONG FILE ---